# 2 Conformal prediction: a guarantee you can actually hold someone to

Notebook 01 built distributions and scored them. It ended somewhere uncomfortable: a
model can produce a confident number that is simply not trustworthy, and calibration —
making the confidence *mean* something — took work, assumptions, and a decision about
which distribution family to impose.

Conformal prediction takes a different route to the same destination.

> Given **any** pretrained model, a held-out calibration set, and a confidence level you
> choose, produce a **set** of labels that contains the truth at least 90% of the time.
> No distributional assumption, no retraining, no requirement that the model be any good.

That is a strange claim on first reading — a guarantee that does not care whether the
model is calibrated, or even whether it is accurate. This notebook builds it in about
fifteen lines, checks the guarantee holds, and then finds the assumption it *does* depend
on and breaks it deliberately.

Read alongside `references/papers.md` → Angelopoulos & Bates,
[arXiv:2107.07511](https://arxiv.org/abs/2107.07511).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from notebooktester import param
from torch import nn
from vectormesh import VectorCache
from vectormesh.components import NeuralNet

BATCH = 128
EPOCHS = param(20, test=1)

cache = VectorCache.from_hub("pttrn-io/dtd-dinov2-small")
d = cache.dataset.shuffle(seed=0).with_format("numpy")
X = torch.tensor(np.asarray(d["embed"], dtype="float32"))
y = torch.tensor(np.asarray(d["label"]))
N_CLASSES = int(y.max()) + 1

# THREE splits, not two. The calibration set is the price of the guarantee.
a, b = int(0.50 * len(X)), int(0.75 * len(X))
Xtr, ytr = X[:a], y[:a]
Xca, yca = X[a:b], y[a:b]
Xte, yte = X[b:], y[b:]
print(f"train {len(Xtr)}   calibration {len(Xca)}   test {len(Xte)}   classes {N_CLASSES}")

**Three splits.** The calibration set is data the model never trained on and never sees
again — it exists only to convert the model's scores into a threshold. That is the price
of the guarantee, and it is the whole cost of the method.

DTD is the right dataset for this: 47 classes and around 0.77 accuracy, so the model is
genuinely unsure a lot of the time. On a saturated task every prediction set would be a
singleton and there would be nothing to see.

In [ ]:
torch.manual_seed(0)
model = NeuralNet(X.shape[1], N_CLASSES)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for _ in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr) - 1, BATCH):
        idx = perm[i:i + BATCH]
        opt.zero_grad()
        loss_fn(model(Xtr[idx]), ytr[idx]).backward()
        opt.step()
model.eval()


def probs(Xa, T=1.0):
    """Softmax scores. `T` is a temperature knob we abuse later on purpose."""
    with torch.no_grad():
        return torch.softmax(model(Xa) / T, dim=-1).numpy()


print(f"ordinary top-1 accuracy on test: {(probs(Xte).argmax(1) == yte.numpy()).mean():.4f}")

## 2.1 The whole method

Three steps, and the only subtle part is the quantile.

1. **Score.** On the calibration set, for each item compute how badly the model treated
   the *true* label: `s = 1 - p(true class)`. A confident correct prediction scores near
   0; a confident mistake scores near 1.
2. **Threshold.** Take the `(1-α)` quantile of those scores, call it `q̂`. Read it as:
   *"on data like this, the true label almost always scores below `q̂`."*
3. **Predict a set.** For a new input, include every class whose probability is at least
   `1 - q̂`.

The finite-sample correction in step 2 — `⌈(n+1)(1-α)⌉ / n` rather than plain `1-α` — is
what turns "approximately" into a guarantee for a calibration set of finite size `n`.

In [ ]:
def calibrate(alpha, Xc=Xca, yc=yca, T=1.0):
    """Return the conformal threshold q-hat for a target coverage of 1 - alpha."""
    p = probs(Xc, T)
    scores = 1.0 - p[np.arange(len(yc)), yc.numpy()]     # how badly the truth scored
    n = len(scores)
    q_level = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(scores, q_level, method="higher"))


def predict_sets(qhat, Xa, T=1.0):
    return probs(Xa, T) >= (1 - qhat)


def evaluate(qhat, Xa, ya, T=1.0):
    sets = predict_sets(qhat, Xa, T)
    covered = sets[np.arange(len(ya)), ya.numpy()]
    return float(covered.mean()), float(sets.sum(1).mean())

## 2.2 Does it hold?

The claim is a number, so check the number. Ask for four different confidence levels and
measure what actually comes back on the test split.

In [ ]:
ALPHAS = param([0.20, 0.10, 0.05, 0.01], test=[0.10])
rows = []
for al in ALPHAS:
    q = calibrate(al)
    cov, size = evaluate(q, Xte, yte)
    rows.append((1 - al, cov, size))
    print(f"asked for {1 - al:.0%}   got {cov:.4f}   average set size {size:5.2f}")

**It holds, at every level.** Ask for 80% and you get about 80%; ask for 99% and you get
about 99%. Nothing here was tuned to make that work — the threshold came straight off the
calibration scores.

The second column is where the actual information lives. Higher confidence is not free:
it is *bought* with larger sets. At 80% the model usually names one class; at 99% it hands
you a list of ten. That is the honest shape of the trade, and it is visible in a way that
"the model said 0.87" never is.

In [ ]:
lv = [r[0] for r in rows]
plt.figure(figsize=(9, 3.4))
plt.subplot(1, 2, 1)
plt.plot(lv, [r[1] for r in rows], "o-", label="empirical")
plt.plot(lv, lv, "--", color="grey", label="nominal")
plt.xlabel("requested coverage"); plt.ylabel("achieved"); plt.legend(); plt.grid(alpha=.3)
plt.subplot(1, 2, 2)
plt.plot(lv, [r[2] for r in rows], "s-", color="tab:orange")
plt.xlabel("requested coverage"); plt.ylabel("average set size"); plt.grid(alpha=.3)
plt.suptitle("the guarantee, and what it costs")
plt.tight_layout(); plt.show()

## 2.3 The part that should bother you

The guarantee made no reference to the model being calibrated. So break the calibration on
purpose and watch what happens.

Dividing the logits by a temperature `T` makes the model absurdly overconfident (`T` small)
or hopelessly timid (`T` large) without changing a single prediction — the ranking of
classes is untouched, only the numbers on top of it move.

In [ ]:
print(f"{'T':>6s} {'mean top-1 confidence':>23s} {'coverage':>10s} {'set size':>10s}")
for T in param([0.25, 1.0, 4.0], test=[1.0]):
    conf = probs(Xte, T).max(1).mean()
    q = calibrate(0.10, T=T)
    cov, size = evaluate(q, Xte, yte, T=T)
    print(f"{T:6.2f} {conf:23.3f} {cov:10.4f} {size:10.2f}")

One model reports average confidence around 0.97 and another around 0.31. **Both give
about 90% coverage.**

This is the thing worth taking away from the notebook. Conformal prediction never asks the
model's numbers to *mean* anything — it only uses them to rank classes, and the calibration
set supplies the meaning. A model whose confidence is pure fiction still yields a valid
prediction set, because the fiction is measured on the calibration data and corrected for.

Miscalibration does not cost you validity. It costs you **efficiency**: a badly scaled
model needs wider sets to reach the same coverage.

## 2.4 Set size is the signal

An interval is only useful if it is narrow when it can be and wide when it must be. Split
the test set by how many labels conformal returned, and check ordinary accuracy within
each group.

In [ ]:
q = calibrate(0.10)
sizes = predict_sets(q, Xte).sum(1)
correct = probs(Xte).argmax(1) == yte.numpy()

print(f"{'set size':>12s} {'items':>7s} {'top-1 accuracy':>16s}")
for lo, hi in [(0, 0), (1, 1), (2, 3), (4, 10), (11, N_CLASSES)]:
    m = (sizes >= lo) & (sizes <= hi)
    if m.sum():
        label = f"{lo}" if lo == hi else f"{lo}-{hi}"
        print(f"{label:>12s} {int(m.sum()):7d} {correct[m].mean():16.3f}")

A clean, monotone signal. When conformal returns a single label the model is right about
93% of the time; when it returns four or more, closer to a third.

**The set size is a per-input difficulty estimate that you get for free**, and unlike a
softmax score it is on a scale you can act on. "Route the multi-label cases to a human" is
a policy you can write down, staff, and cost — which is the subject of notebook 03.

Sets can also come out **empty**, though not at this confidence level — you will see it in
§2.5, where the average size drops below 1. An empty set is not a bug: it is the method
saying *none of these labels clears the bar*, a refusal to answer that an argmax can never
produce.

## 2.5 The assumption, and what happens when it fails

Nothing is free. Conformal needs **exchangeability**: the calibration data and the data
you deploy on have to be drawn from the same distribution.

That is a real condition, so let us break it in a way that can be measured. Rank the
classes by how well the model handles them, calibrate on the *easy* half, and then deploy
on the *hard* half — a model meeting inputs harder than the ones it was calibrated on,
which is the single most common way this fails in production.

In [ ]:
pc = probs(Xca).argmax(1)
per_class = {k: float((pc[yca.numpy() == k] == k).mean())
             for k in range(N_CLASSES) if (yca.numpy() == k).sum() > 0}
ranked = sorted(per_class, key=lambda k: per_class[k], reverse=True)
EASY = set(ranked[: len(ranked) // 2])

ca_easy = np.isin(yca.numpy(), list(EASY))
te_easy = np.isin(yte.numpy(), list(EASY))
print(f"easy half: mean per-class accuracy {np.mean([per_class[k] for k in EASY]):.3f}")
print(f"hard half: mean per-class accuracy "
      f"{np.mean([v for k, v in per_class.items() if k not in EASY]):.3f}\n")

q_easy = calibrate(0.10, Xc=Xca[ca_easy], yc=yca[ca_easy])
cov_s, size_s = evaluate(q_easy, Xte[te_easy], yte[te_easy])
cov_d, size_d = evaluate(q_easy, Xte[~te_easy], yte[~te_easy])
cov_a, size_a = evaluate(calibrate(0.10), Xte, yte)

print(f"{'':34s} {'coverage':>9s} {'set size':>10s}   (asked for 0.90)")
print(f"{'exchangeable: all -> all':34s} {cov_a:9.4f} {size_a:10.2f}")
print(f"{'exchangeable: easy -> easy':34s} {cov_s:9.4f} {size_s:10.2f}")
print(f"{'VIOLATED:     easy -> hard':34s} {cov_d:9.4f} {size_d:10.2f}")

**Coverage collapses.** You asked for 90% and got roughly 65% — a promise broken by
25 points.

Now look at the set-size column, because that is the genuinely alarming part. **It barely
moves.** The failing system does not produce suspiciously wide sets, or empty ones, or any
other visible symptom. It produces confident-looking output at the usual size, while
silently covering the truth two times in three instead of nine times in ten.

So the guarantee is real and it is conditional, and the condition is about your *data*
rather than your model. Which gives the practical rule:

> Conformal tells you nothing about inputs unlike the ones you calibrated on. If your
> deployment distribution can drift, the calibration set has to drift with it — and
> monitoring coverage on fresh labelled data is not optional, because nothing else in the
> system will tell you it has gone wrong.

Not every shift does this. Splitting the label space *at random* instead of by difficulty
leaves coverage almost untouched (0.888 against a nominal 0.90), because the two halves are
about equally hard. It is specifically **shift in difficulty** — the calibration set being
systematically easier than what you deploy on — that breaks the guarantee. Exercise E6.

## What you now know

- **Any model, any confidence level, a guarantee** — for the price of a calibration split
  and about fifteen lines.
- **Validity does not require calibration.** A model with nonsense confidences still gets
  valid sets; miscalibration costs set width, not correctness.
- **Set size is a free per-input difficulty score**, and empty sets are a legitimate
  refusal to answer.
- **Exchangeability is the whole assumption**, and violating it fails *silently*.

## Exercises

**E1 — the calibration set is not free.** Re-run §2.2 with 100, 250 and the full
calibration split. How small can it get before the achieved coverage becomes unreliable?
Repeat each size a few times with different random calibration subsets — the point is the
spread, not the mean.

**E2 — a better score function.** We used `1 - p(true class)`. The APS variant instead
accumulates sorted probabilities until the true class is reached. Implement it, and
compare average set size at the same coverage. Validity should be unchanged; efficiency is
where score functions differ.

**E3 — conformal on a weak encoder.** Repeat the whole notebook on `pttrn-io/dtd-resnet-18`
(about 15 points less accurate, per lesson 3). Coverage should still hold. Predict what
happens to the average set size before you run it, and say why.

**E4 — regression, not classification.** Notebook 01 built a quantile head on a time
series. Conformalise it: use the calibration residuals to widen the predicted interval
until it actually covers 90%, and compare the result with the quantile head's own nominal
interval. Which one keeps its promise?

**E6 — which shifts matter?** Redo §2.5 splitting the classes at random rather than by
difficulty, and confirm coverage survives. Then construct a third split of your own that
you predict *will* break it, and check. What property of a shift determines whether
conformal notices?

**E5 — detect the failure in §2.5.** Coverage collapsed with no change in set size, so set
size is not the alarm. Given only *unlabelled* deployment inputs, what would you monitor to
notice this before a customer does? Implement one candidate and test whether it separates
the easy-half from the hard-half inputs.